# Blending Classifier

Blending trains base models on a training partition and trains a meta-model on their predictions for a separate validation partition. A final test split is kept for evaluation.

In [ ]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression

X, y = make_classification(
    n_samples=800, n_features=20, n_informative=10,
    n_redundant=2, random_state=42
)
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.25, stratify=y_temp, random_state=42
)

base_models = [
    ("rf", RandomForestClassifier(n_estimators=200, max_depth=5, random_state=42)),
    ("gb", GradientBoostingClassifier(n_estimators=150, max_depth=3, random_state=42))
]
for name, model in base_models:
    model.fit(X_train, y_train)
    print(f"{name} validation accuracy: {accuracy_score(y_val, model.predict(X_val)):.3f}")

val_features = np.column_stack([model.predict_proba(X_val)[:, 1] for _, model in base_models])
meta_model = LogisticRegression(max_iter=1000, random_state=42)
meta_model.fit(val_features, y_val)

test_features = np.column_stack([model.predict_proba(X_test)[:, 1] for _, model in base_models])
blend_pred = meta_model.predict(test_features)
print("Blending test accuracy:", accuracy_score(y_test, blend_pred))
print("\nClassification report:\n", classification_report(y_test, blend_pred))

## Interpretation and cautions

The validation holdout trains the meta-model; it is not the final test set. Keep test data out of both base-model fitting and meta-model fitting. Blending is similar to stacking, but stacking commonly trains the meta-model from cross-validated predictions rather than a single validation holdout.